# 10) Maths-3 Lagrangians and KKT conditions — Exercises

**Goals:** equal incremental cost, a KKT checker that certifies (or rejects) a dispatch, λ-iteration with limits, how the price grows with demand, and nodal prices on a congested line — including the value of extra line capacity, checked with PyPSA.

The setup cell holds the lecture's three units and two-bus network. Costs C(P) = a·P + b·P² in €/h, marginal costs in €/MWh.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [ ]:
import logging
import warnings

import numpy as np
import pandas as pd
import pypsa
from scipy.optimize import brentq, minimize

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

A = np.array([20.0, 30.0, 40.0])        # EUR/MWh
B = np.array([0.020, 0.010, 0.015])     # EUR/MWh^2
P_MAX = np.array([400.0, 500.0, 300.0]) # MW

# two-bus network (lecture): cheap unit at A, expensive unit and load at B
A_A, B_A, A_B, B_B, LOAD = 20.0, 0.020, 35.0, 0.030, 700.0


def total_cost(P, a=A, b=B):
    return float(np.sum(a * np.asarray(P) + b * np.asarray(P) ** 2))

### Warm-ups

**1. Equal incremental cost without limits**

Write `unconstrained_dispatch(D, a, b)` returning (λ, P) from the closed form of the lecture (all units at marginal cost λ, limits ignored).

In [ ]:
def unconstrained_dispatch(D: float, a=A, b=B) -> tuple[float, np.ndarray]:
    """Price (EUR/MWh) and outputs (MW) with equal marginal costs, ignoring limits."""
    ...  # YOUR CODE HERE

In [ ]:
lam, P = unconstrained_dispatch(600)
assert np.isclose(P.sum(), 600) and np.allclose(A + 2 * B * P, lam), "balance and equal marginal costs"
assert np.isclose(unconstrained_dispatch(800)[0], 38.1538, atol=1e-4), "the lecture's infeasible solution"

**2. A KKT checker**

Write `kkt_violation(P, lam, D)` returning the largest violation (≥ 0) of the four KKT conditions for the three-unit problem, with the multipliers computed from P and λ as in the lecture (μ⁺ = λ − C' at the maximum, μ⁻ = C' − λ at zero, otherwise 0). A true optimum gives ≈ 0; anything else gives a clear positive number.

In [ ]:
def kkt_violation(P, lam: float, D: float, tol: float = 1e-6) -> float:
    """Largest violation of stationarity, feasibility, dual feasibility (and complementarity by construction)."""
    ...  # YOUR CODE HERE

In [ ]:
assert kkt_violation([400, 400, 0], 38.0, 800) < 1e-9, "the lecture's optimum"
assert kkt_violation([800 / 3] * 3, 38.0, 800) > 1, "an equal split is not optimal"
assert kkt_violation(unconstrained_dispatch(800)[1], unconstrained_dispatch(800)[0], 800) > 1, "violates the limits"
assert kkt_violation([400, 400, 0], 37.0, 800) > 0.5, "right dispatch, wrong price: a multiplier turns negative"

### Core

**3. λ-iteration with limits**

Write `lambda_iteration(D)` returning (λ, P): find the price at which the clipped outputs meet demand (bisection or `brentq`). Raise `ValueError` if D exceeds total capacity. Check it at 1,100 MW — predict the answer first — and compare with SLSQP.

In [ ]:
def lambda_iteration(D: float) -> tuple[float, np.ndarray]:
    """Economic dispatch with limits: price (EUR/MWh) and outputs (MW)."""
    ...  # YOUR CODE HERE

In [ ]:
lam, P = lambda_iteration(1100)
assert np.isclose(lam, 46.0) and np.allclose(P, [400, 500, 200]), "units 1 and 2 at their limits, unit 3 marginal"
assert kkt_violation(P, lam, 1100) < 1e-6
ref = minimize(total_cost, np.full(3, 1100 / 3), method="SLSQP", bounds=list(zip([0.0] * 3, P_MAX)),
               constraints=[{"type": "eq", "fun": lambda x: x.sum() - 1100}], options={"ftol": 1e-12, "maxiter": 500})
assert np.allclose(ref.x, P, atol=1e-3)
try:
    lambda_iteration(1300)
    raise AssertionError("1,300 MW exceeds the 1,200 MW capacity")
except ValueError:
    pass
print(f"D = 1100 MW: price {lam:.2f} EUR/MWh, P = {P} MW")

**4. How the price grows with demand**

Write `price_curve(demands)` returning a DataFrame (index: demand in MW) with the price, the three outputs, and the finite-difference marginal cost of serving demand, (C*(D + 0.5) − C*(D − 0.5)). The two price columns must agree — the envelope theorem.

In [ ]:
def price_curve(demands) -> pd.DataFrame:
    """Price, dispatch and finite-difference marginal cost for each demand level."""
    ...  # YOUR CODE HERE

In [ ]:
curve = price_curve(range(200, 1150, 100))
assert curve.price_EUR_MWh.is_monotonic_increasing, "more demand never lowers the price"
assert np.allclose(curve.price_EUR_MWh, curve.finite_difference, atol=1e-3), "envelope theorem"
curve.round(2)

**5. Interpretation**

In Maths-2 (linear costs) the price jumped between plateaus; here it rises smoothly with demand, except where its *slope* changes. Explain both behaviours with the KKT conditions, and say what the slope changes in your curve correspond to physically.

*Your answer here.*

### Challenge

**6. What is a line upgrade worth?**

For the two-bus network, write `two_bus(F)` returning the bus prices (λ_A, λ_B), the flow and the total generation cost for a line limit F. Solve it **with the KKT conditions** rather than a numerical optimiser: find the uncongested solution (both buses at one price), clip the flow at F if it exceeds the limit, and compute each bus price from its unit's stationarity condition. Then:

1. check the result against PyPSA for F = 300 MW;
2. show by finite differences that −dC*/dF (the cost saved per extra MW of line capacity) equals λ_B − λ_A, the line's multiplier.

In [ ]:
def two_bus(F: float) -> dict:
    """Bus prices (EUR/MWh), flow (MW) and total cost (EUR/h) for line limit F."""
    ...  # YOUR CODE HERE


def two_bus_pypsa(F: float) -> tuple[np.ndarray, float]:
    """Bus prices and flow from PyPSA (quadratic costs), for checking."""
    n = pypsa.Network()
    n.set_snapshots([0])
    n.add("Bus", ["A", "B"])
    n.add("Generator", "gen_A", bus="A", p_nom=800, marginal_cost=A_A, marginal_cost_quadratic=B_A)
    n.add("Generator", "gen_B", bus="B", p_nom=800, marginal_cost=A_B, marginal_cost_quadratic=B_B)
    n.add("Load", "load", bus="B", p_set=LOAD)
    n.add("Line", "A-B", bus0="A", bus1="B", x=0.1, r=0.0, s_nom=F)
    n.optimize(solver_name="highs", log_to_console=False)
    return n.buses_t.marginal_price.iloc[0].to_numpy(), float(n.lines_t.p0.iloc[0, 0])

In [ ]:
r300 = two_bus(300)
prices, flow = two_bus_pypsa(300)
assert np.allclose(prices, [r300["price_A"], r300["price_B"]], rtol=1e-4) and np.isclose(flow, r300["flow"], rtol=1e-4)

rows = []
for F in [200, 300, 400, 500, 600]:
    r = two_bus(F)
    value_fd = two_bus(F - 0.5)["cost"] - two_bus(F + 0.5)["cost"]          # cost saved per extra MW
    rows.append({"line_limit_MW": F, "price_A": r["price_A"], "price_B": r["price_B"],
                 "mu_line": r["price_B"] - r["price_A"], "value_of_1MW_more": value_fd})
upgrade = pd.DataFrame(rows).set_index("line_limit_MW")
assert np.allclose(upgrade.mu_line, upgrade.value_of_1MW_more, atol=1e-3), "envelope theorem for the line"
assert upgrade.mu_line.is_monotonic_decreasing and np.isclose(upgrade.loc[600, "mu_line"], 0, atol=1e-6)
upgrade.round(2)

**7. Interpretation**

A planner can add 100 MW to the line at an annualised cost of 40,000 €/MW·year. Using your table and assuming this hour is typical of all 8,760 hours (it is not — but take it as a first estimate), is the upgrade from 300 to 400 MW worth it? Why is the line's multiplier, and not the average price difference, the right number to use, and why does its value fall as the line grows?

*Your answer here.*